# 01 Validate Input PPE Sources

Validate the new PPE v2 input layout for the four-class detector:

- `0 = person`
- `1 = helmet`
- `2 = vest`
- `3 = cleaning_coverall`

This notebook is intentionally small. It validates input data and writes only the two reports needed before splitting.


## Purpose

Input data is now organized by source lane:

```text
data/input/open_source/images/     -> train only later
data/input/open_source/labels/

data/input/factory_source/images/  -> train + validation later
data/input/factory_source/labels/

data/input/test_source/images/     -> final untouched test set later
data/input/test_source/labels/
```

Notebook 01 does **not** merge, copy, split, augment, train, or create generated datasets. Later notebooks will create runtime outputs under `data/generated/` when needed.


## Setup

Find the `v2_pipeline` root, add it to `sys.path`, and import the small validation helpers. Keeping the heavy checks in `src/` makes this notebook easier to read and safer to maintain.


In [1]:
from pathlib import Path
import sys

import yaml
from IPython.display import display


# Notebook files can be launched from different working directories depending
# on Jupyter/VS Code. This helper walks upward until it finds the v2 pipeline
# root, so all later paths can stay project-relative instead of absolute.
def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline folder from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not find v2_pipeline root. Run this notebook from inside the repository."
    )


V2_ROOT = find_v2_root(Path.cwd()).resolve()

# Add v2_pipeline to Python's import path so notebook cells can import reusable
# code from src/ without installing this module as a package.
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

# Imports from src happen after sys.path is updated. The noqa comment keeps
# linters quiet about this notebook-friendly import order.
from src.dataset.validate_yolo_dataset import (  # noqa: E402
    build_source_summary,
    validate_input_sources,
)

print(f"v2 root: {V2_ROOT}")


v2 root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


## Load Configuration

The configs are the single source of truth for paths, image extensions, and class IDs. This keeps the notebook free from hardcoded absolute paths.


In [2]:
# Load the dataset and class configs separately so path settings and the class
# taxonomy can evolve independently.
config_path = V2_ROOT / "configs" / "dataset_config.yaml"
class_config_path = V2_ROOT / "configs" / "class_names.yaml"

with config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)

with class_config_path.open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

# These are the only source folders Notebook 01 reads. Open-source data is
# validated here but will be train-only later; factory data will feed train/val;
# test_source remains untouched until final evaluation.
source_dirs = {
    "open_source": V2_ROOT / dataset_config["input"]["open_source"],
    "factory_source": V2_ROOT / dataset_config["input"]["factory_source"],
    "test_source": V2_ROOT / dataset_config["input"]["test_source"],
}

# Reports are safe to create here because they are generated artifacts and are
# ignored by Git. Data/generated is not created by this notebook.
reports_validation_dir = V2_ROOT / dataset_config["reports"]["validation"]
reports_validation_dir.mkdir(parents=True, exist_ok=True)

# YAML can load mapping keys as strings, so convert class IDs back to int before
# passing them into validation. This avoids subtle class-id comparison bugs.
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}
class_ids = set(class_names)
valid_image_extensions = {
    ext.lower() for ext in dataset_config["valid_image_extensions"]
}

print("Input sources:")
for source_name, source_dir in source_dirs.items():
    print(f"- {source_name}: {source_dir}")

print("Class map:")
for class_id, class_name in class_names.items():
    print(f"- {class_id}: {class_name}")


Input sources:
- open_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\open_source
- factory_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\factory_source
- test_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\test_source
Class map:
- 0: person
- 1: helmet
- 2: vest
- 3: cleaning_coverall


## Validate Sources

The validator checks image readability, matching image-label pairs, class IDs, YOLO row shape, numeric normalized coordinates, positive box size, and image-boundary safety. Hidden placeholder files such as `.gitkeep` are ignored.


In [3]:
# validate_input_sources is read-only: it scans images/labels, validates YOLO
# rows, and returns a DataFrame. It does not copy files or create split folders.
validation_df = validate_input_sources(
    source_dirs=source_dirs,
    class_ids=class_ids,
    image_extensions=valid_image_extensions,
    # Keep this strict for PPE input data. If you intentionally add background
    # images later, document that change before setting this to True.
    allow_empty_labels=False,
)

# Keep one complete validation report on disk. Invalid and warning rows are
# displayed below by filtering this DataFrame in memory, so separate CSVs would
# add clutter without adding new information.
validation_report_path = reports_validation_dir / "validation_report.csv"
validation_df.to_csv(validation_report_path, index=False)

print(f"Saved validation report: {validation_report_path}")
print(f"Validation rows: {len(validation_df)}")
display(validation_df.head(20))


Saved validation report: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\validation_report.csv
Validation rows: 905


,source,base_name,image_name,label_name,image_path,label_path,status,errors,warnings,image_width,image_height,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall
0,open_source,ppe_0008,ppe_0008.jpg,ppe_0008.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,960,640,8,4,0,4,0
1,open_source,ppe_0016,ppe_0016.jpg,ppe_0016.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,5760,3840,18,7,7,4,0
2,open_source,ppe_0018,ppe_0018.jpg,ppe_0018.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,800,600,4,2,2,0,0
3,open_source,ppe_0023,ppe_0023.jpg,ppe_0023.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,540,304,17,6,6,5,0
4,open_source,ppe_0037,ppe_0037.jpg,ppe_0037.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,800,566,7,5,2,0,0
5,open_source,ppe_0038,ppe_0038.jpg,ppe_0038.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,1026,579,11,4,4,3,0
6,open_source,ppe_0048,ppe_0048.jpg,ppe_0048.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,900,600,27,10,9,8,0
7,open_source,ppe_0051,ppe_0051.jpg,ppe_0051.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,1280,720,3,1,1,1,0
8,open_source,ppe_0052,ppe_0052.jpg,ppe_0052.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,1140,500,6,2,2,2,0
9,open_source,ppe_0055,ppe_0055.jpg,ppe_0055.txt,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,valid,,Duplicate base name appears across sources; la...,950,637,2,1,1,0,0


## Compact Source Summary

This is the only summary artifact Notebook 01 saves. It is enough to confirm whether each source lane has valid images and whether the four PPE classes are present before moving to the split notebook.


In [4]:
# This compact summary is the only extra artifact from Notebook 01. It gives a
# quick source-by-source count of valid rows, invalid rows, warnings, and the
# four object classes, including cleaning_coverall.
source_summary_df = build_source_summary(
    validation_df=validation_df,
    source_names=tuple(source_dirs.keys()),
)

source_summary_path = reports_validation_dir / "source_summary.csv"
source_summary_df.to_csv(source_summary_path, index=False)

print(f"Saved source summary: {source_summary_path}")
display(source_summary_df)


Saved source summary: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\validation\source_summary.csv


,source,total_rows,valid_rows,invalid_rows,warning_rows,total_objects,num_person,num_helmet,num_vest,num_cleaning_coverall
0,open_source,180,180,0,38,1751,722,636,393,0
1,factory_source,637,637,0,38,4115,1703,1050,1234,128
2,test_source,88,88,0,0,352,158,97,67,30


## Review Issues In Memory

The full report is already saved, so this cell only displays invalid and warning rows for quick notebook review. It does not create extra CSV files.


In [5]:
# Review problems in the notebook without writing more files. If something is
# invalid, fix the source image/label pair and rerun Notebook 01.
if validation_df.empty:
    print(
        "No input samples found yet. Add files under data/input/{open_source,factory_source,test_source}/images and labels."
    )
else:
    # Invalid rows block downstream splitting because the image-label pair is
    # missing, unreadable, or not valid YOLO format.
    invalid_df = validation_df.loc[validation_df["status"].ne("valid")].copy()

    # Warning rows are review prompts. For example, duplicate basenames across
    # source lanes can be handled later, but exact duplicates may deserve a look.
    warning_df = validation_df.loc[validation_df["warnings"].fillna("").ne("")].copy()

    if invalid_df.empty:
        print("No invalid rows found.")
    else:
        print(f"Invalid rows found: {len(invalid_df)}")
        display(
            invalid_df[
                ["source", "base_name", "image_name", "label_name", "errors"]
            ].head(50)
        )

    if warning_df.empty:
        print("No warning rows found.")
    else:
        print(f"Warning rows found: {len(warning_df)}")
        display(warning_df[["source", "base_name", "image_name", "warnings"]].head(50))


No invalid rows found.
Warning rows found: 76


,source,base_name,image_name,warnings
0,open_source,ppe_0008,ppe_0008.jpg,Duplicate base name appears across sources; la...
1,open_source,ppe_0016,ppe_0016.jpg,Duplicate base name appears across sources; la...
2,open_source,ppe_0018,ppe_0018.jpg,Duplicate base name appears across sources; la...
3,open_source,ppe_0023,ppe_0023.jpg,Duplicate base name appears across sources; la...
4,open_source,ppe_0037,ppe_0037.jpg,Duplicate base name appears across sources; la...
5,open_source,ppe_0038,ppe_0038.jpg,Duplicate base name appears across sources; la...
6,open_source,ppe_0048,ppe_0048.jpg,Duplicate base name appears across sources; la...
7,open_source,ppe_0051,ppe_0051.jpg,Duplicate base name appears across sources; la...
8,open_source,ppe_0052,ppe_0052.jpg,Duplicate base name appears across sources; la...
9,open_source,ppe_0055,ppe_0055.jpg,Duplicate base name appears across sources; la...


## Split Policy Reminder

Notebook 01 does not create splits. Notebook 02 should use this source policy:

- `train`: valid `open_source` samples plus the training portion of valid `factory_source` samples.
- `val`: validation portion of valid `factory_source` samples only.
- `test`: valid `test_source` samples only.

This keeps validation and test factory-focused while still using open-source data to expand training perspective.


## Final Checklist

Before moving to Notebook 02:

- `reports/validation/validation_report.csv` was saved.
- `reports/validation/source_summary.csv` was saved.
- Invalid rows displayed above have been reviewed and fixed if needed.
- Class `3 = cleaning_coverall` appears correctly in the report as `num_cleaning_coverall`.
- No merged dataset, generated split, augmented data, experiment data, training run, or model weight was created by this notebook.
